# 03 · 암종별 오분류 원인 가설 검증 — 변이 정보 표현 실험

**연구 질문:** 변이 유무(0/1)만 사용하는 모델에 구체적인 `유전자=변이값` 정보를 추가하면 암종별 오분류가 줄어드는가?

**공식 지표:** Macro F1. **보조 지표:** 클래스별 F1 및 특정 암종 쌍 혼동 빈도.

**누출 방지:** Train에서 만든 기존 학습/검증 분할만 사용한다. Test CSV는 이 노트북에서 읽지 않는다. 해시 기반 변환은 전체 데이터의 빈도나 통계를 학습하지 않는다.

> 결과 해석 주의: 변이 문자열이 복합값/동의어로 저장된 경우, 문자열이 서로 다른 것이 반드시 생물학적으로 다른 변이를 뜻하지 않는다. 이는 특징 표현 실험이며 임상적 검증이 아니다.

## 1. 필수 라이브러리 및 경로

In [ ]:
# 필요하면 VS Code 터미널에서 실행: python -m pip install numpy pandas scipy scikit-learn matplotlib
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from IPython.display import display

ROOT = Path.cwd()
OUTPUTS = ROOT / 'outputs'
TRAIN_CSV = ROOT / 'data' / 'train(1).csv'
RESULTS = ROOT / 'experiment_03'
RESULTS.mkdir(exist_ok=True)
assert (OUTPUTS / 'split_indices.npz').exists(), '기존 outputs 폴더를 프로젝트 폴더에 넣어주세요.'
assert TRAIN_CSV.exists(), 'data/train(1).csv를 넣어주세요.'
print('원본:', TRAIN_CSV, '전처리:', OUTPUTS)

## 2. 기존 분할 및 변이 이진 특징 불러오기

In [ ]:
X_binary = sparse.load_npz(OUTPUTS/'X_train_binary.npz').tocsr().astype(np.float32)
y = np.load(OUTPUTS/'y_train.npy', allow_pickle=True).astype(str)
splits = np.load(OUTPUTS/'split_indices.npz')
fit_idx, valid_idx = splits['fit_idx'], splits['valid_idx']
with open(OUTPUTS/'gene_columns.json',encoding='utf-8') as f:
    gene_cols = json.load(f)
raw = pd.read_csv(TRAIN_CSV, low_memory=False)
assert len(raw) == len(y) == X_binary.shape[0]
assert all(col in raw.columns for col in gene_cols)
assert np.array_equal(raw['SUBCLASS'].astype(str).to_numpy(), y)
assert not set(fit_idx) & set(valid_idx)
print('샘플 / 유전자:',X_binary.shape, '\n학습 / 검증:',len(fit_idx),len(valid_idx))

## 3. `유전자=변이값` 해시 특징 생성

WT와 빈 값은 변이 특징으로 사용하지 않는다. FeatureHasher는 학습 단계가 없는 고정 변환이다. **Test를 사용하지 않는다.**

In [ ]:
def patient_variant_tokens(frame, cols):
    # 각 환자별로 실제 변이가 있는 유전자의 (유전자, 값) 문자열만 산출
    for row in frame[cols].itertuples(index=False, name=None):
        tokens = []
        for gene, value in zip(cols, row):
            if pd.isna(value):
                continue
            value = str(value).strip()
            if value and value.upper() != 'WT':
                tokens.append(f'{gene}={value}')
        yield tokens

hasher = FeatureHasher(n_features=2**18, input_type='string', alternate_sign=False, dtype=np.float32)
X_variant = hasher.transform(patient_variant_tokens(raw, gene_cols)).tocsr()
X_combined = sparse.hstack([X_binary, X_variant],format='csr')
print('이진 특징:',X_binary.shape,'변이 문자열 해시 특징:',X_variant.shape,'합친 특징:',X_combined.shape)
print('추가 변이 특징 nnz:',X_variant.nnz)

## 4. 같은 모델 설정으로 두 표현 방식 비교

In [ ]:
def build_model():
    return SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42,
                         max_iter=1500, tol=1e-3)

predictions={}
rows=[]
for name, X in [('binary_only', X_binary), ('binary_plus_variant', X_combined)]:
    model = build_model()
    model.fit(X[fit_idx],y[fit_idx])
    pred = model.predict(X[valid_idx])
    predictions[name]=pred
    rows.append({'feature_representation':name,
                 'macro_f1':f1_score(y[valid_idx],pred,average='macro'),
                 'accuracy':accuracy_score(y[valid_idx],pred)})
comparison=pd.DataFrame(rows)
display(comparison)
comparison.to_csv(RESULTS/'model_comparison.csv',index=False,encoding='utf-8-sig')

## 5. 암종별 F1 변화 비교

In [ ]:
reports={}
for name,pred in predictions.items():
    report = pd.DataFrame(classification_report(y[valid_idx],pred,output_dict=True,zero_division=0)).T
    reports[name]=report
    report.to_csv(RESULTS/f'class_report_{name}.csv',encoding='utf-8-sig')
classes=sorted(set(y))
per_class=pd.DataFrame({
    'class':classes,
    'support':[int(reports['binary_only'].loc[c,'support']) for c in classes],
    'f1_binary':[reports['binary_only'].loc[c,'f1-score'] for c in classes],
    'f1_variant':[reports['binary_plus_variant'].loc[c,'f1-score'] for c in classes],
})
per_class['f1_delta']=per_class.f1_variant-per_class.f1_binary
per_class=per_class.sort_values('f1_delta')
display(per_class)
per_class.to_csv(RESULTS/'per_class_f1_change.csv',index=False,encoding='utf-8-sig')

## 6. 특정 혼동 쌍 변화 검증

**주의:** 예시 쌍은 기존 Baseline에서 오류가 많았던 쌍이다. 여러 암종 쌍을 비교하는 경우 표본 수와 다중 비교 문제에 유의한다.

In [ ]:
pairs=[('KIPAN','KIRC'),('KIRC','KIPAN'),('GBMLGG','LGG'),('LGG','GBMLGG'),('BRCA','OV'),('OV','BRCA')]
truth=y[valid_idx]
conf=[]
for true,pred_cls in pairs:
    row={'true':true,'predicted':pred_cls,'true_support':int((truth==true).sum())}
    for name,pred in predictions.items():
        row[f'errors_{name}']=int(((truth==true)&(pred==pred_cls)).sum())
    row['error_delta']=row['errors_binary_plus_variant']-row['errors_binary_only']
    conf.append(row)
conf_df=pd.DataFrame(conf)
display(conf_df)
conf_df.to_csv(RESULTS/'target_confusions.csv',index=False,encoding='utf-8-sig')

## 7. 환자별 예측과 오류 기록

In [ ]:
record=pd.DataFrame({'ID':raw.iloc[valid_idx]['ID'].to_numpy() if 'ID' in raw.columns else valid_idx,
                     'true':truth,'pred_binary':predictions['binary_only'],
                     'pred_variant':predictions['binary_plus_variant']})
record['binary_correct']=record.true==record.pred_binary
record['variant_correct']=record.true==record.pred_variant
record['improved']=~record.binary_correct & record.variant_correct
record['regressed']=record.binary_correct & ~record.variant_correct
record.to_csv(RESULTS/'validation_predictions_comparison.csv',index=False,encoding='utf-8-sig')
print('개선된 환자:',int(record.improved.sum()),'새롭게 틀린 환자:',int(record.regressed.sum()))
display(record.head())

## 연구 기록 템플릿

- **문제:** 어떤 암종에서 혼동이 집중됐는가?
- **원인 가설:** 이진화가 변이 종류 차이를 소실시켜 해당 암종을 구분하기 어려울 수 있다.
- **선택한 전략:** `gene=variant` 토큰을 해시 특징으로 추가.
- **검증 방법:** 동일 분할, 동일 SGD, Macro F1·암종별 F1·혼동 쌍 비교.
- **해석:** 전체 점수 개선 여부뿐 아니라 목표 암종의 오류 감소와 다른 암종의 성능 저하도 함께 보고.
- **한계:** 한 번의 검증 분할로 인과적 원인 확정 불가. 추후 그룹 기반 교차검증과 반복 실험 필요.

**다음 실험:** 해당 암종을 중심으로 클래스 가중치 개선 또는 암종 차별 유전자 특징 선택 실험을 별도로 실시해 표본 수·변이 특징 표현 효과를 구분한다.